# 課程18：使用加密收據確保AI代理安全

## 實作筆記本

本筆記本將引導完成四個練習：

1. <strong>簽署您的第一張收據</strong>以紀錄代理工具調用並驗證它。
2. <strong>竄改收據</strong>並觀看驗證失敗的情況。
3. <strong>建立三張收據鏈</strong>並確認鏈的完整性。
4. **封裝 Microsoft Agent Framework 工具調用**，使每個動作皆發出收據。

所有加密原語皆從維護良好的函式庫引入（`pynacl` 用於 Ed25519，`jcs` 用於 RFC 8785 標準化 JSON，Python 標準庫 `hashlib` 用於 SHA-256）。收據邏輯本身是純 Python，您可以閱讀並修改。

請依序運行各儲存格。每個章節簡短且自成一體。


## 安裝設定

安裝這兩個依賴套件。兩者皆為寬鬆授權（Apache-2.0 / MIT）。


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## 輔助工具

這兩個輔助工具負責 base64url 編碼（不帶填充）和對任意物件進行 SHA-256 雜湊。它們讓筆記本的其他部分專注於收據邏輯本身。


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## 第一部分：簽署您的第一個收據

想像我們為 **Contoso Travel** 的代理人剛剛為客戶查詢了從悉尼到洛杉磯的航班。我們想把這個工具調用記錄為簽署的收據，以便未來的審核員可以在不需要信任我們的情況下驗證它。

### 步驟 1.1：產生簽署金鑰

在生產環境中，代理人的簽署金鑰會存放在硬體安全模組 (HSM)、Azure Key Vault 或類似的受保護存儲中。這課程中我們在記憶體中產生新的金鑰。


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### 步驟 1.2：構建收據負載

負載包含我們希望收據證明的一切：誰執行了操作，使用了什麼工具，帶了什麼參數，回傳了什麼，依照什麼政策，以及何時執行。我們對參數和結果進行哈希處理，而不是內嵌它們，以免收據洩漏敏感內容。


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### 步驟 1.3：簽署及組裝收據

三個步驟：

1. 使用 JCS 對有效負載進行規範化，使兩個實作生成相同邏輯收據會產生位元完全相同的位元組。
2. 使用 SHA-256 對規範化的位元組進行雜湊。
3. 使用 Ed25519 私鑰對雜湊值進行簽署。

然後將簽名附加到原始有效負載中，以生成最終收據。


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### 步驟 1.4：驗證收條

驗證是反轉該過程。我們剝除簽名，重新計算規範哈希，然後根據收條中的公鑰檢查簽名。

進行此驗證的審計員只需收條本身。無需呼叫任何服務，無需查詢密鑰目錄，也不需信任。


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

你應該會見到 `Receipt is valid: True`。代理程式已經產生它的第一個密碼學簽名審計紀錄。


## 第二部分：篡改收據

收據的整個目的就是讓它們具備防篡改的特性。讓我們來證明一下。

我們將修改收據中恰好一個字符，並觀察驗證失敗的情況。


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### 剛剛發生了什麼？

當我們更改 `policy_id` 時，正規字節也隨之改變。這些字節的 SHA-256 雜湊值改變了。對原始雜湊值簽署的簽章，現在不再符合新的雜湊值。驗證正確地返回 `False`。

除非攻擊者擁有私鑰，否則沒有辦法修改任何收據欄位且仍然通過驗證。只要私鑰保存在金鑰庫中並且公鑰已公開，篡改是不可能隱藏的。

自己試試看：修改上方儲存格中的 `tool_name`、`agent_id` 或 `timestamp`，然後重新執行。每次改動都會產生無效收據。


## 第 3 節：將收據串連在一起

一張收據只保護一個操作。大多數代理人會執行多個操作。為了讓整個序列具有防篡改性，我們在新收據的有效負載中包含前一張收據的哈希值，從而將每張收據連接到前一張收據。

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

如果有人刪除或重新排序收據，鏈條會在該點斷裂。任何後續收據的驗證都會失敗，因為其 `previous_receipt_hash` 不再與其前身的實際哈希值相符。


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

現在透過竄改中間的收據來打破鏈接，然後重新驗證。被竄改的收據會驗證簽名失敗，而下一張收據會驗證鏈接失敗（因為它的 `previous_receipt_hash` 已不再與被修改的中間收據的哈希值相符）。


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

收據 0 仍然通過驗證（因為它沒有被修改，且沒有前置收據需要依賴）。收據 1 因為我們更改了 `tool_args_hash` 而未能通過其簽名檢查。收據 2 因為其 `previous_receipt_hash` 是基於原本（現在已修改的）收據 1 計算，未能通過鏈接檢查。

即使攻擊者重新簽署修改後的收據 1（沒有私鑰他們無法做到），收據 2 中的鏈接不匹配仍會揭露篡改。要隱藏這種改動，攻擊者必須從修改點開始重新簽署每一張收據，這需要持有私鑰。


## 第四部分：以回執簽署包裝代理工具調用

在實際部署中，您不希望每個代理作者都需要記得調用 `make_receipt`。您希望每次工具調用時都自動進行回執簽署。

這裡是最簡單的範例：一個包裝類，它接受任何可調用的工具函數並返回一個帶有回執輸出版本的函數。這可以適配任何代理框架，包括 Microsoft 代理框架（`agent_framework.foundry`）。

如果您尚未設定 Microsoft Foundry 專案，下面的本地模擬示例仍然展示了這個模式。


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### 與 Microsoft Agent Framework 整合

上述的 `ReceiptedTool` 包裝器是框架無關的。要在使用 Microsoft Agent Framework 建立的代理中使用它，請將包裝的函數註冊為一個工具。以下是一個示意（你會將模擬替換為真實的 Microsoft Foundry 工具註冊）：

```python
# 顯示整合形態的偽代碼。
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="你係 Contoso 旅遊代理 ...",
#     tools=[receipted_lookup],   # 包裝咗嘅工具，唔係原始函數
# )
# response = agent.run("搵下六月由悉尼飛去洛杉磯嘅航班。")
#
# # 運行後，代理使用嘅每個工具呼叫都有簽署嘅收據：
# audit_chain = receipted_lookup.receipts
```

代理框架不需要知道任何關於收據的事情。收據簽署環繞在工具周圍，而不是嵌入到框架中。這就是如何在不重寫代理的情況下，向現有代理代碼添加來源記錄的方法。


## 回顧與延伸挑戰

你已經：

- 生成了一對 Ed25519 密鑰。
- 建立並簽署了一個代理工具呼叫的收據。
- 只使用公鑰在離線環境驗證了該收據。
- 篡改了一個收據並觀察到驗證失敗。
- 創建了一個包含三個收據的哈希鏈序列。
- 篡改了鏈中間的部分，並觀察到簽名失敗和鏈接失敗。
- 將工具函數封裝起來自動簽署收據。

**延伸挑戰。** 在收據結構中新增一個 `request_id` 欄位（一個用於分散式追蹤的 UUID）。更新 `make_receipt` 以包含該欄位，並確認收據仍可端到端驗證。之後在簽署後修改該欄位並確認驗證失敗。這將迫使你內化瞭解規範編碼的每一個位元組如何影響簽名。

**重要界限。** 收據證明三件事且只有這三件事：歸屬（此密鑰簽署了該內容）、完整性（自簽署後內容沒有更改）、以及排序（此收據是在另一收據之後產生）。它們不證明代理的行動是正確的，不證明名為 `policy_id` 的政策確實被評估，也不證明代理遵守所有規則。收據是一個基礎。治理是你建立於其上的系統。

帶著這個界限再讀一次本課程的 README。團隊在使用收據時最常見的錯誤是假設「我們有收據」就代表「我們有治理」。事實並非如此。收據使代理行為可審計，但不代表其正確。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
